# Classificação Interativa - Distúrbios do Sono

Notebook para o **Google Colab** baseado em `predict.py`.

Treina o mesmo modelo de árvore de decisão e permite classificar **novos casos** informando os dados de uma pessoa.

**Como usar no Colab:** execute as células na ordem, faça o upload do `Sleep_health_and_lifestyle_dataset.csv` e, na última célula, edite os valores de exemplo e rode para obter a previsão.

## 1. Instalar e importar bibliotecas

In [ ]:
# !pip install -q scikit-learn pandas

import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

## 2. Carregar o dataset (upload no Colab)

In [ ]:
from google.colab import files

uploaded = files.upload()  # selecione Sleep_health_and_lifestyle_dataset.csv
csv_name = list(uploaded.keys())[0]
print('Arquivo carregado:', csv_name)

## 3. Pré-processamento (mesma lógica do train.py)

In [ ]:
df = pd.read_csv(csv_name, keep_default_na=True)
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')
df = df.drop(columns=['Person ID'])
df[['Systolic', 'Diastolic']] = (
    df['Blood Pressure'].str.split('/', expand=True).astype(int)
)
df = df.drop(columns=['Blood Pressure'])

numeric_features = [
    'Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level',
    'Stress Level', 'Heart Rate', 'Daily Steps', 'Systolic', 'Diastolic',
]
categorical_features = ['Gender', 'Occupation', 'BMI Category']

X = df[numeric_features + categorical_features]
y = df['Sleep Disorder']

## 4. Treinar o modelo

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)
modelo = Pipeline([
    ('prep', preprocessor),
    ('clf', DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)),
])
modelo.fit(X, y)
print('Modelo treinado.')

# Valores válidos para consulta
print('Gêneros:', sorted(df['Gender'].unique()))
print('Profissões:', sorted(df['Occupation'].unique()))
print('IMC:', sorted(df['BMI Category'].unique()))

## 5. Função de previsão

Edite os valores abaixo e execute a célula para classificar um novo caso. A função retorna o distúrbio previsto e as probabilidades por classe.

In [ ]:
def prever(genero, idade, profissao, duracao_sono, qualidade_sono,
           atividade, estresse, bmi, sistolica, diastolica, bpm, passos):
    entrada = pd.DataFrame([{
        'Gender': genero,
        'Age': idade,
        'Occupation': profissao,
        'Sleep Duration': duracao_sono,
        'Quality of Sleep': qualidade_sono,
        'Physical Activity Level': atividade,
        'Stress Level': estresse,
        'BMI Category': bmi,
        'Systolic': sistolica,
        'Diastolic': diastolica,
        'Heart Rate': bpm,
        'Daily Steps': passos,
    }])
    previsao = modelo.predict(entrada)[0]
    probabilidades = modelo.predict_proba(entrada)[0]
    classes = modelo.classes_
    print('=== Resultado ===')
    print(f'Distúrbio do sono previsto: {previsao}')
    print('Probabilidades por classe:')
    for classe, prob in sorted(zip(classes, probabilidades), key=lambda x: -x[1]):
        print(f'  {classe:>12}: {prob:.2%}')
    return previsao

## 6. Classificar um novo caso (formulário interativo)

Agora você **não precisa mais digitar** os valores: use os **menus suspensos** para Gênero, Profissão e IMC, e os **controles deslizantes** para os números. Depois clique em **"Prever distúrbio"**.

**O que é cada campo:**
- **Gênero** (menu): `Male` ou `Female`
- **Idade** (anos): idade da pessoa
- **Profissão** (menu): uma das profissões presentes no dataset
- **Sono (h)**: horas de sono por noite
- **Qualid. sono**: nota de 1 a 10
- **Ativ. física**: nível de atividade física registrado no dataset
- **Estresse**: nível de 1 a 10
- **IMC** (menu): categoria de peso (`Normal`, `Overweight`, `Obese`)
- **Sistólica / Diastólica**: pressão arterial (ex.: 126/83)
- **BPM (repouso)**: batimentos por minuto em repouso
- **Passos/dia**: passos diários

> As opções dos menus são carregadas automaticamente a partir do próprio dataset, então você sempre verá valores válidos.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# --- Opções disponíveis (extraídas automaticamente do dataset) ---
generos = sorted(df['Gender'].unique())
profissoes = sorted(df['Occupation'].unique())
imcs = sorted(df['BMI Category'].unique())

# --- Campos categóricos: selecione em vez de digitar ---
w_genero = widgets.Dropdown(options=generos, value=generos[0], description='Gênero:')
w_profissao = widgets.Dropdown(options=profissoes, value=profissoes[0], description='Profissão:')
w_bmi = widgets.Dropdown(options=imcs, value=imcs[0], description='IMC:')

# --- Campos numéricos: controles deslizantes com faixas do dataset ---
w_idade = widgets.IntSlider(min=int(df['Age'].min()), max=int(df['Age'].max()), value=35, description='Idade (anos):')
w_duracao = widgets.FloatSlider(min=4.0, max=10.0, step=0.1, value=6.1, description='Sono (h):')
w_qualidade = widgets.IntSlider(min=1, max=10, value=6, description='Qualid. sono:')
w_atividade = widgets.IntSlider(min=int(df['Physical Activity Level'].min()), max=int(df['Physical Activity Level'].max()), value=42, description='Ativ. física:')
w_estresse = widgets.IntSlider(min=1, max=10, value=6, description='Estresse:')
w_sistolica = widgets.IntSlider(min=80, max=160, value=126, description='Sistólica:')
w_diastolica = widgets.IntSlider(min=50, max=110, value=83, description='Diastólica:')
w_bpm = widgets.IntSlider(min=50, max=110, value=77, description='BPM (repouso):')
w_passos = widgets.IntSlider(min=int(df['Daily Steps'].min()), max=int(df['Daily Steps'].max()), value=4200, description='Passos/dia:')

botao = widgets.Button(description='Prever distúrbio', button_style='success')

def ao_clicar(b):
    prever(
        genero=w_genero.value,
        idade=w_idade.value,
        profissao=w_profissao.value,
        duracao_sono=w_duracao.value,
        qualidade_sono=w_qualidade.value,
        atividade=w_atividade.value,
        estresse=w_estresse.value,
        bmi=w_bmi.value,
        sistolica=w_sistolica.value,
        diastolica=w_diastolica.value,
        bpm=w_bpm.value,
        passos=w_passos.value,
    )

botao.on_click(ao_clicar)

formulario = widgets.VBox([
    widgets.HTML('<b>Preencha os dados e clique em "Prever distúrbio":</b>'),
    widgets.HBox([w_genero, w_profissao]),
    widgets.HBox([w_bmi, w_idade]),
    widgets.HBox([w_duracao, w_qualidade]),
    widgets.HBox([w_atividade, w_estresse]),
    widgets.HBox([w_sistolica, w_diastolica]),
    widgets.HBox([w_bpm, w_passos]),
    botao,
])
display(formulario)